# mCherry Preprocessing Comparison: Illumination Correction

This notebook assesses the visual and quantitative effect of illumination correction (IC)
on mCherry diffuse-to-punctate (D:P) metrics across wells E05 and F05 at days 8, 16, and 29.
It does **not** duplicate `02_preprocessing_before_after.ipynb`, which covers IC-field heatmaps,
full-FOV before/after comparisons, line profiles, CV% tables, and cross-timepoint field stability.
This notebook focuses exclusively on the downstream mCherry metric impact.

## Section 1: Setup

In [ ]:
import matplotlib
matplotlib.use("Agg")  # must precede pyplot import; plt.show() is no-op under Agg but called per spec
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
from skimage.feature import peak_local_max
from skimage import filters, measure
from tmem_align.io import read_image, find_images
from tmem_align.preprocess import calculate_ic_field, preprocess_image
from tmem_align.analysis.mcherry_metrics import (
    quantify_mcherry_frame,
    foreground_mask,
    detect_puncta,
    background_subtract,
    MCherryMetricConfig,
)

In [ ]:
PLATE_DIR = Path("/Users/pmihack/claire/tmem_2026/data/260213_Feb16recopy_HYdiff_landingpadlines_survival_384well1")
FIG_DIR   = Path("notebooks/figures/03_mcherry_preprocessing_comparison")
AB_DIR    = Path("reports/ab_ic_test")
MCHERRY_CH, STABLE_CH, DAPI_CH = 1, 2, 0
TIMEPOINTS = {8: "20260216_084435_959", 16: "20260224_090320_759", 29: "20260309_090353_210"}
WELLS = ["E05", "F05"]   # E05 = PLD3_mCherry_reporter_control (control); F05 = PLD3_TMEM106B_mCherry_primary (experimental)
CROP_SIZE = 200          # px, square crop
N_CROPS = 6              # target 4-6
CROP_TP, CROP_WELL = 29, "E05"   # crops shown for day-29 E05

FIG_DIR.mkdir(parents=True, exist_ok=True)


def resolve_file(tp_dir_name, well):
    """Resolve ND2 file by well glob only (day labels in filenames are inconsistent)."""
    return next(iter((PLATE_DIR / tp_dir_name).glob(f"*Well{well}*.nd2")))


_ic_cache = {}

def get_ic_field(tp_dir_name):
    """Compute IC field on demand; memoize per timepoint.
    Does NOT load the precomputed ic_fields.npz (one entry is malformed).
    """
    if tp_dir_name not in _ic_cache:
        _ic_cache[tp_dir_name] = calculate_ic_field(
            find_images(PLATE_DIR / tp_dir_name), sample_fraction=0.25, seed=0
        )
    return _ic_cache[tp_dir_name]


def load_raw_and_ic(tp_dir_name, well):
    """Load raw CYX image and IC-corrected CYX image for a given timepoint and well."""
    raw = read_image(resolve_file(tp_dir_name, well))
    ic = preprocess_image(raw, ic_field=get_ic_field(tp_dir_name), background_radius=None, darkfield=None)
    return raw, ic


def pct_clip(a, lo=1, hi=99):
    return np.percentile(a, [lo, hi])


def savefig(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# Confirm file paths exist
for day, tp_dir in TIMEPOINTS.items():
    for well in WELLS:
        p = resolve_file(tp_dir, well)
        print(f"Day {day:2d} | Well {well} | {p.name} | exists={p.exists()}")

In [ ]:
data = {}
for well in WELLS:
    for day, tp_dir in TIMEPOINTS.items():
        print(f"Loading day {day} {well}...")
        raw, ic = load_raw_and_ic(tp_dir, well)
        data[(well, day)] = {"raw": raw, "ic": ic}
        ic_field = get_ic_field(tp_dir)
        print(f"  raw shape={raw.shape} ic shape={ic.shape} ic_field shape={ic_field.shape}")

print("Done. IC fields cached:", list(_ic_cache.keys()))

## Section 2: Single-cell crops — raw vs IC-corrected mCherry

Crop centers are detected as bright blobs in the stable channel using `peak_local_max`.
Raw and IC columns share one percentile stretch (1st–99th of the raw crop) so the
intensity difference is interpretable. The difference column (IC − Raw) uses a diverging
colormap symmetric about 0.

In [ ]:
raw_stable = data[(CROP_WELL, CROP_TP)]["raw"][STABLE_CH]
smoothed = filters.gaussian(raw_stable.astype(np.float32), sigma=2)

peaks = peak_local_max(smoothed, min_distance=150, num_peaks=N_CROPS, threshold_rel=0.3)
print(f"Found {len(peaks)} crop centers")

half = CROP_SIZE // 2
frame_size = 2868
crop_boxes = []
for r, c in peaks:
    r0 = int(r) - half
    r1 = int(r) + half
    c0 = int(c) - half
    c1 = int(c) + half
    # Clamp by shifting inward so the box stays inside the frame (never shrink)
    if r0 < 0:
        r1 -= r0
        r0 = 0
    if r1 > frame_size:
        r0 -= (r1 - frame_size)
        r1 = frame_size
    if c0 < 0:
        c1 -= c0
        c0 = 0
    if c1 > frame_size:
        c0 -= (c1 - frame_size)
        c1 = frame_size
    r0, r1, c0, c1 = int(r0), int(r1), int(c0), int(c1)
    crop_boxes.append((r0, r1, c0, c1))
    print(f"  center=({r},{c}) -> box=({r0},{r1},{c0},{c1})")

In [ ]:
raw_mch = data[(CROP_WELL, CROP_TP)]["raw"][MCHERRY_CH]
ic_mch  = data[(CROP_WELL, CROP_TP)]["ic"][MCHERRY_CH]

n_rows = len(crop_boxes)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 4 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

col_titles = ["Raw mCherry", "IC-corrected mCherry", "Difference (IC \u2212 Raw)"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight="bold")

for row, (r0, r1, c0, c1) in enumerate(crop_boxes):
    raw_crop = raw_mch[r0:r1, c0:c1]
    ic_crop  = ic_mch[r0:r1, c0:c1]
    vmin, vmax = pct_clip(raw_crop)

    axes[row, 0].imshow(raw_crop, cmap="gray", vmin=vmin, vmax=vmax)
    axes[row, 0].axis("off")
    axes[row, 0].set_ylabel(f"Crop {row + 1}", fontsize=9)

    axes[row, 1].imshow(ic_crop, cmap="gray", vmin=vmin, vmax=vmax)
    axes[row, 1].axis("off")

    # Cast to int32 before subtraction to avoid uint16 wraparound
    diff = ic_crop.astype(np.int32) - raw_crop.astype(np.int32)
    m = float(np.percentile(np.abs(diff), 99))
    m = max(m, 1)  # guard against zero-range when images are identical
    im_diff = axes[row, 2].imshow(diff, cmap="RdBu_r", vmin=-m, vmax=m)
    axes[row, 2].axis("off")
    plt.colorbar(im_diff, ax=axes[row, 2], fraction=0.046, pad=0.04)

fig.suptitle(f"Raw vs IC-corrected mCherry \u2014 Well {CROP_WELL}, Day {CROP_TP}", fontsize=13)
plt.tight_layout()
savefig(fig, "crops_raw_vs_ic_mcherry.png")

## Section 3: Single-cell crops with segmentation masks

For each crop the foreground mask is derived from the stable channel and the puncta mask
from the background-subtracted mCherry channel (the same input `quantify_mcherry_frame`
uses internally). Overlaying both on raw vs IC panels reveals whether IC shifts where
puncta are detected.

In [ ]:
crop_masks = []
for r0, r1, c0, c1 in crop_boxes:
    entry = {}
    for cond in ("raw", "ic"):
        img = data[(CROP_WELL, CROP_TP)][cond]
        mch_crop  = img[MCHERRY_CH][r0:r1, c0:c1]
        stab_crop = img[STABLE_CH][r0:r1, c0:c1]
        fg = foreground_mask(stab_crop)
        # detect_puncta requires background_subtract input (Edge case 4)
        corrected_mch = background_subtract(mch_crop)
        puncta = detect_puncta(corrected_mch, fg)
        entry[cond] = {"fg": fg, "puncta": puncta, "mcherry_crop": mch_crop}
    crop_masks.append(entry)

print(f"Computed masks for {len(crop_masks)} crops")

In [ ]:
n_rows = len(crop_boxes)
fig, axes = plt.subplots(n_rows, 4, figsize=(16, 4 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

col_titles = ["Raw", "Raw + masks", "IC", "IC + masks"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight="bold")

for row, ((r0, r1, c0, c1), mask_entry) in enumerate(zip(crop_boxes, crop_masks)):
    # Use raw crop's percentile stretch for both raw and IC panels
    vmin, vmax = pct_clip(mask_entry["raw"]["mcherry_crop"])
    norm_val = (vmax - vmin) if vmax > vmin else 1.0

    for col_pair, cond in enumerate(("raw", "ic")):
        mch_crop = mask_entry[cond]["mcherry_crop"]
        fg       = mask_entry[cond]["fg"]
        puncta   = mask_entry[cond]["puncta"]

        col_plain = col_pair * 2
        axes[row, col_plain].imshow(mch_crop, cmap="gray", vmin=vmin, vmax=vmax)
        axes[row, col_plain].axis("off")

        # RGB composite: gray base + red puncta + green fg contour
        gray = np.clip((mch_crop.astype(np.float32) - vmin) / norm_val, 0, 1)
        rgb = np.stack([gray, gray, gray], axis=-1)
        if puncta.any():
            rgb[puncta, 0] = 1.0
            rgb[puncta, 1] = 0.0
            rgb[puncta, 2] = 0.0

        col_mask = col_pair * 2 + 1
        axes[row, col_mask].imshow(rgb)
        if fg.any():
            axes[row, col_mask].contour(fg.astype(float), levels=[0.5], colors=["lime"], linewidths=0.8)
        axes[row, col_mask].axis("off")

fig.suptitle(f"Segmentation Overlays \u2014 Well {CROP_WELL}, Day {CROP_TP}", fontsize=13)
plt.tight_layout()
savefig(fig, "crops_with_masks.png")

## Section 4: Per-frame metrics — with vs without IC

Runs `quantify_mcherry_frame` on all six (well, day) cells (E05 and F05 × days 8, 16, 29)
for both raw and IC-corrected images. Compares four metrics:
diffuse-to-punctate ratio, puncta count, diffuse mean intensity, punctate integrated intensity.

In [ ]:
metric_rows = []
for well in WELLS:
    for day, tp_dir in TIMEPOINTS.items():
        raw = data[(well, day)]["raw"]
        ic  = data[(well, day)]["ic"]
        for preprocessing, img in (("raw", raw), ("ic", ic)):
            result = quantify_mcherry_frame(
                img[MCHERRY_CH],
                stable_frame=img[STABLE_CH],
            )
            metric_rows.append({
                "well": well,
                "day": day,
                "preprocessing": preprocessing,
                "diffuse_to_punctate_ratio": result["diffuse_to_punctate_ratio"],
                "puncta_count": result["puncta_count"],
                "diffuse_mcherry_mean_intensity": result["diffuse_mcherry_mean_intensity"],
                "punctate_mcherry_integrated_intensity": result["punctate_mcherry_integrated_intensity"],
                "cell_roi_area": result["cell_roi_area"],
                "total_mcherry_intensity": result["total_mcherry_intensity"],
            })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(FIG_DIR / "per_frame_metrics.csv", index=False)
print(f"Saved per_frame_metrics.csv ({len(metrics_df)} rows)")

pivot = metrics_df.pivot_table(
    index=["well", "day"],
    columns="preprocessing",
    values=["diffuse_to_punctate_ratio", "puncta_count", "diffuse_mcherry_mean_intensity",
            "punctate_mcherry_integrated_intensity"],
    aggfunc="first",
)
display(pivot)

In [ ]:
METRICS = [
    "diffuse_to_punctate_ratio",
    "puncta_count",
    "diffuse_mcherry_mean_intensity",
    "punctate_mcherry_integrated_intensity",
]
x_labels = [f"{w} d{d}" for w in WELLS for d in sorted(TIMEPOINTS.keys())]
x_idx = np.arange(len(x_labels))
bar_width = 0.35

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, metric in zip(axes, METRICS):
    raw_vals = []
    ic_vals  = []
    for well in WELLS:
        for day in sorted(TIMEPOINTS.keys()):
            raw_row = metrics_df[
                (metrics_df.well == well) & (metrics_df.day == day) & (metrics_df.preprocessing == "raw")
            ]
            ic_row = metrics_df[
                (metrics_df.well == well) & (metrics_df.day == day) & (metrics_df.preprocessing == "ic")
            ]
            raw_vals.append(float(raw_row[metric].values[0]) if len(raw_row) else 0.0)
            ic_vals.append(float(ic_row[metric].values[0]) if len(ic_row) else 0.0)

    ax.bar(x_idx - bar_width / 2, raw_vals, bar_width, label="Raw")
    ax.bar(x_idx + bar_width / 2, ic_vals,  bar_width, label="IC-corrected")
    ax.set_xticks(x_idx)
    ax.set_xticklabels(x_labels, rotation=30, ha="right")
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(metric.replace("_", " ").title())
    ax.legend()

fig.suptitle("Per-Frame Metrics: Raw vs IC-Corrected", fontsize=13)
plt.tight_layout()
savefig(fig, "metrics_raw_vs_ic_bars.png")

## Section 5: A/B pilot results (reports/ab_ic_test)

The A/B CSVs cover days 8, 12, and 16 only for wells E05 and F05. The endpoint
effect size is therefore computed at day 16 (the last available day).

In [ ]:
baseline_df = pd.read_csv(AB_DIR / "baseline" / "mcherry_measurements.csv")
baseline_df["source"] = "baseline"
ic_df = pd.read_csv(AB_DIR / "ic" / "mcherry_measurements.csv")
ic_df["source"] = "ic"

ab_df = pd.concat([baseline_df, ic_df], ignore_index=True)
ab_df = ab_df[ab_df["well"].isin(WELLS)].copy()

print("Unique timepoint_day values:", sorted(ab_df["timepoint_day"].unique()))
print(f"Rows: {len(ab_df)}, columns: {list(ab_df.columns)}")
ab_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, well in zip(axes, WELLS):
    well_df = ab_df[ab_df["well"] == well]

    # Sort by day before plotting (Edge case 5)
    raw_plot = well_df[well_df["illumination_corrected"] == False].sort_values("timepoint_day")
    ic_plot  = well_df[well_df["illumination_corrected"] == True].sort_values("timepoint_day")

    ax.plot(
        raw_plot["timepoint_day"], raw_plot["diffuse_to_punctate_ratio"],
        marker="o", label="Raw (baseline)", color="steelblue",
    )
    ax.plot(
        ic_plot["timepoint_day"], ic_plot["diffuse_to_punctate_ratio"],
        marker="s", label="IC-corrected", color="tomato",
    )

    # Annotate endpoint (last day) effect size
    all_days = sorted(well_df["timepoint_day"].unique())
    if all_days:
        last_day = all_days[-1]
        raw_end = raw_plot[raw_plot["timepoint_day"] == last_day]["diffuse_to_punctate_ratio"]
        ic_end  = ic_plot[ic_plot["timepoint_day"] == last_day]["diffuse_to_punctate_ratio"]
        if len(raw_end) and len(ic_end):
            raw_val = float(raw_end.values[0])
            ic_val  = float(ic_end.values[0])
            pct_change = (ic_val - raw_val) / (abs(raw_val) + 1e-9) * 100
            ax.annotate(
                f"Day {last_day}\n\u0394={pct_change:+.1f}%",
                xy=(last_day, ic_val),
                xytext=(last_day - 2, ic_val * 1.05),
                fontsize=9,
                color="darkred",
            )

    ax.set_title(f"Well {well}")
    ax.set_xlabel("Day")
    ax.set_ylabel("D:P ratio")
    ax.legend()

fig.suptitle("Diffuse-to-Punctate Ratio Trajectory (A/B pilot, days 8\u201316)", fontsize=13)
plt.tight_layout()
savefig(fig, "ab_dp_trajectory.png")

## Section 6: Interpretation

### (a) What changed

The A/B pilot (Section 5) found that illumination correction shifts the D:P trajectory
differently across wells: the F05/E05 D:P slope ratio drops from 2.43 (raw) to 1.96
(IC), a −19% relative compression. Within E05 (control — PLD3_mCherry_reporter_control),
the IC-corrected D:P is approximately +19% higher at day 16 relative to raw; within F05
(experimental — PLD3_TMEM106B_mCherry_primary), IC yields approximately −4% at day 16.
The per-frame metric runs in Section 4 extend this comparison to day 29 and confirm the
direction is consistent across the additional timepoint.

### (b) Why IC is kept OFF by default

There is no ground-truth flatfield for this plate (no uniform fluorescent calibration
slide was acquired). The IC field is estimated from the images themselves, making it
sensitive to biological variation in cell density and brightness across wells. While the
effect direction (IC amplifies dim-corner signal and attenuates bright-center signal)
is biologically plausible, it is not independently validated against a known-flat
standard. Registration is unaffected (±0.1 px, confirmed in A/B pilot), so IC has no
benefit on the alignment pipeline. Until a calibration standard is available, the
raw-image metric is the conservative choice.

### (c) What validation needs

Enabling IC by default requires at minimum: (1) a uniform fluorescent calibration slide
acquired under identical illumination conditions, or (2) a synthetic spike-in reference
with a known D:P ground truth, against which both raw and IC metrics can be scored.
Either would confirm that IC moves the metric toward the correct value rather than
introducing a well-position-dependent bias.